In [ ]:
# ============================================================
# Retail Intelligence Brasil
# Notebook.......: 06_bronze_regioes_imediatas
# Camada.........: Bronze
# Fonte..........: IBGE - API Localidades
# Objetivo.......: Persistir as regioes geograficas imediatas na camada Bronze.
# Autor..........: Cristhina Freire
# Data...........: 17/06/2026
# ============================================================



In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import requests
import json
from datetime import datetime



In [ ]:
# ============================================================
# 2. CONFIGURAÇÃO
# ============================================================

CATALOG = "retail_intelligence"
SCHEMA = "bronze"

TABLE_NAME = "ibge_regioes_imediatas_raw"

FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

BASE_URL = "https://servicodados.ibge.gov.br/api/v1/localidades/regioes-imediatas"

TIMEOUT = 60

CURRENT_DATE = datetime.now().strftime("%Y-%m-%d")
CURRENT_TIME = datetime.now().strftime("%Y%m%d_%H%M%S")

VOLUME_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/landing/"
    f"ibge/regioes_imediatas/{CURRENT_DATE}"
)

FILE_NAME = f"regioes_imediatas_{CURRENT_TIME}.json"

FILE_PATH = f"{VOLUME_PATH}/{FILE_NAME}"

print("=" * 60)
print("BRONZE - IBGE REGIÕES IMEDIATAS")
print("=" * 60)



In [ ]:
# ============================================================
# 3. EXTRAÇÃO
# ============================================================

response = requests.get(BASE_URL, timeout=TIMEOUT)
response.raise_for_status()

landing_data = response.json()

print(f"Registros encontrados: {len(landing_data)}")



In [ ]:
# ============================================================
# 4. PERSISTÊNCIA LANDING
# ============================================================

dbutils.fs.mkdirs(VOLUME_PATH)

dbutils.fs.put(
    FILE_PATH,
    json.dumps(landing_data, ensure_ascii=False, indent=2),
    overwrite=True
)

print(f"JSON salvo em:\n{FILE_PATH}")



In [ ]:
# ============================================================
# 5. LEITURA LANDING
# ============================================================

landing_df = (
    spark.read
         .option("multiline", "true")
         .json(FILE_PATH)
)



In [ ]:
# ============================================================
# 6. PERSISTÊNCIA BRONZE
# ============================================================

(
    landing_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(FULL_TABLE_NAME)
)



In [ ]:
# ============================================================
# 7. INSPEÇÃO
# ============================================================

bronze_df = spark.table(FULL_TABLE_NAME)

print(f"Total: {bronze_df.count()}")

bronze_df.printSchema()

display(bronze_df)



In [ ]:
# ============================================================
# 8. ENCERRAMENTO
# ============================================================

print("=" * 60)
print("Pipeline finalizado com sucesso.")
print("=" * 60)